# Mammalian Family Classification from COI-5P DNA Barcodes

**ISA 3 – AI Lab | Data Science and Data Engineering**  
**Students:** Prashant Naik (2607) and Karan Haldankar (2612)

## Project objective
Predict the **mammalian family** from a COI-5P DNA barcode sequence, then investigate individual false-positive and false-negative predictions.

### Important design decisions
- Keep **all family classes**, including families with very few records. Do not impose a minimum class-frequency cutoff.
- Use DNA-derived features as the primary model inputs. Do **not** use `genus`, `species`, `order`, `family`-related taxonomy, process IDs, or location as predictive features.
- Retain specimen/taxonomic metadata separately for later error investigation.
- Exact duplicate sequences must not be split between training and testing. Identical sequences with conflicting family labels are ambiguous and will be flagged and excluded from supervised modelling, while remaining in the audit table.
- A family represented by only one unique sequence cannot be evaluated on a held-out test set while also being present in training. Such families will remain in training and be explicitly reported as not independently testable.

This notebook is the first reproducible baseline. It reports the actual class distribution before modelling so rare-class limitations are visible.

> **Important:** This corrected version fixes sequence whitespace cleaning and the timing/evaluation cells. Because sequence cleaning can change the data, run the notebook from the beginning (`Kernel` → `Restart Kernel and Run All`) before using any metrics or exported CSV files.


## 1. Imports and configuration
Run this notebook from the folder containing the downloaded BOLD TSV, or update `DATA_PATH` below.

In [ ]:
# If a package is missing, uncomment and run:
# %pip install pandas numpy scikit-learn matplotlib seaborn joblib

from pathlib import Path
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
TEST_SIZE = 0.20

# Change this if your TSV is stored elsewhere.
DATA_PATH = Path("DS-EBACMAMM.tsv")

# The BOLD export has many columns; load only those needed for the first phase.
REQUIRED_COLUMNS = [
    "processid", "sampleid", "family", "genus", "species",
    "nuc", "marker_code", "country/ocean", "province/state",
    "collection_date_start", "habitat", "identification",
]
print("Data file:", DATA_PATH.resolve())
print("File exists:", DATA_PATH.exists())


## 2. Load the BOLD dataset
The notebook checks the required fields and prints the dimensions. If the file is not found, update `DATA_PATH` to the location of the TSV.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Set DATA_PATH to the full path of your downloaded BOLD TSV."
    )

df = pd.read_csv(DATA_PATH, sep="\t", usecols=lambda c: c in REQUIRED_COLUMNS,
                 low_memory=False, dtype={"processid": "string"})
print("Rows:", len(df))
print("Columns loaded:", df.columns.tolist())
display(df.head(3))


## 3. Select mammalian COI-5P records
We select rows where `marker_code` is exactly `COI-5P`, a family label is present, and a nucleotide sequence is available. **No family is removed based on its record count.**

In [ ]:
df["marker_code"] = df["marker_code"].astype("string").str.strip()
df["family"] = df["family"].astype("string").str.strip()

coi = df[
    df["marker_code"].eq("COI-5P")
    & df["family"].notna()
    & df["family"].ne("")
    & df["nuc"].notna()
].copy()

print(f"Usable COI-5P records with family labels: {len(coi):,}")
print(f"Distinct family labels: {coi['family'].nunique():,}")
print(f"Distinct species labels (where supplied): {coi['species'].nunique(dropna=True):,}")


## 4. Clean and validate DNA sequences
For the first baseline, retain canonical DNA bases (`A`, `C`, `G`, `T`) and remove whitespace/gaps. Ambiguous bases are counted and flagged; records containing non-ACGT symbols are not used in the initial k-mer model, but are retained in the audit dataframe. This avoids silently treating ambiguous characters as ordinary nucleotides.

In [ ]:
def clean_sequence(value):
    if pd.isna(value):
        return ""
    # Uppercase and remove all whitespace (spaces, tabs and line breaks).
    seq = re.sub(r"\s+", "", str(value).upper())
    # Remove alignment gap characters.
    seq = seq.replace("-", "").replace(".", "")
    return seq

coi["sequence_raw"] = coi["nuc"].astype("string")
coi["sequence"] = coi["sequence_raw"].map(clean_sequence)
coi["sequence_length"] = coi["sequence"].str.len()
coi["ambiguous_base_count"] = coi["sequence"].str.count(r"[^ACGT]")
coi["is_acgt_only"] = coi["sequence"].str.fullmatch(r"[ACGT]+").fillna(False)

print("Records with only A/C/G/T:", int(coi["is_acgt_only"].sum()))
print("Records containing ambiguous/non-ACGT symbols:", int((~coi["is_acgt_only"]).sum()))
print("Sequence length summary:")
display(coi["sequence_length"].describe())

# This audit table keeps excluded rows and the reason they were not modelled.
coi["audit_status"] = np.where(coi["is_acgt_only"], "eligible_for_baseline", "ambiguous_or_invalid_sequence")
display(coi[["processid", "family", "marker_code", "sequence_length",
             "ambiguous_base_count", "audit_status"]].head())


## 5. Inspect all family classes
This is an important decision point. We keep every family, including singleton and very rare families. The plot uses a logarithmic scale so that small classes remain visible beside very large ones.

In [ ]:
family_counts_all = (
    coi.loc[coi["is_acgt_only"], "family"]
    .value_counts()
    .rename_axis("family")
    .reset_index(name="record_count")
)

print("Number of represented families:", len(family_counts_all))
print("Families with 1 record:", int((family_counts_all.record_count == 1).sum()))
print("Families with fewer than 5 records:", int((family_counts_all.record_count < 5).sum()))
print("Families with fewer than 10 records:", int((family_counts_all.record_count < 10).sum()))
display(family_counts_all)

plt.figure(figsize=(14, 5))
plt.bar(np.arange(len(family_counts_all)), family_counts_all["record_count"])
plt.yscale("log")
plt.xlabel("Family (ordered by record count)")
plt.ylabel("Number of records (log scale)")
plt.title("COI-5P records per mammalian family — all classes retained")
plt.tight_layout()
plt.show()


## 6. Check exact duplicate sequences and conflicting labels
Repeated records with the same sequence and same family can inflate performance if copies enter both splits. We therefore model one row per unique sequence. If an identical sequence has more than one family label, its label is ambiguous; we exclude that sequence from supervised training/testing but keep all source records in `coi` for investigation.

In [ ]:
eligible = coi.loc[coi["is_acgt_only"]].copy()
eligible["sequence"] = eligible["sequence"].astype(str)

seq_family_counts = eligible.groupby("sequence")["family"].nunique()
conflicting_sequences = set(seq_family_counts[seq_family_counts > 1].index)

print("Eligible source records:", len(eligible))
print("Unique sequences:", eligible["sequence"].nunique())
print("Repeated source records beyond first copy:",
      len(eligible) - eligible["sequence"].nunique())
print("Unique sequences associated with conflicting family labels:",
      len(conflicting_sequences))

# Keep one representative per unique sequence only when its family label is unambiguous.
unambiguous = eligible.loc[~eligible["sequence"].isin(conflicting_sequences)].copy()
model_df = (
    unambiguous.sort_values("processid", na_position="last")
    .drop_duplicates(subset=["sequence"], keep="first")
    .reset_index(drop=True)
)

print("Unique, unambiguous sequences for baseline modelling:", len(model_df))
print("Families remaining in model table:", model_df["family"].nunique())
print("Families represented in eligible records but absent after conflict filtering:",
      sorted(set(eligible["family"]) - set(model_df["family"])))


## 7. Create a duplicate-safe split while retaining all possible classes
We split **unique sequences within each family**. Every family with at least two unique sequences gets at least one sequence in train and one in test. A family with only one unique sequence stays in training because it is mathematically impossible to place that one sequence in both sets. Such a family is retained, but its generalisation performance cannot be measured from this single record.

This split prevents exact sequence duplicates from leaking across train and test. It does not guarantee that closely related sequences or records from the same species are separated; a stricter species-grouped evaluation can be added after the baseline.

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
train_indices, test_indices = [], []
not_testable_families = []

for family, group in model_df.groupby("family", sort=True):
    idx = group.index.to_numpy(copy=True)
    rng.shuffle(idx)
    n = len(idx)
    if n == 1:
        train_indices.extend(idx.tolist())
        not_testable_families.append(family)
    else:
        n_test = max(1, int(round(n * TEST_SIZE)))
        n_test = min(n_test, n - 1)  # always leave at least one training example
        test_indices.extend(idx[:n_test].tolist())
        train_indices.extend(idx[n_test:].tolist())

train_df = model_df.loc[train_indices].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)
test_df = model_df.loc[test_indices].sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

train_families = set(train_df["family"])
test_families = set(test_df["family"])
print("Train sequences:", len(train_df))
print("Test sequences:", len(test_df))
print("Families in training:", len(train_families))
print("Families represented in test:", len(test_families))
print("Families not independently testable (one unique sequence):", len(not_testable_families))
print(sorted(not_testable_families))

assert train_families == set(model_df["family"]), "A family is missing from training."
assert set(train_df["sequence"]).isdisjoint(set(test_df["sequence"])), "Exact sequence leakage detected."
display(pd.DataFrame({
    "train_count": train_df["family"].value_counts(),
    "test_count": test_df["family"].value_counts()
}).fillna(0).astype(int).sort_index())


## 8. Extract DNA k-mer features
A k-mer is a consecutive substring of length *k*. For example, `ATGCTA` produces 4-mers `ATGC`, `TGCT`, and `GCTA`. We use a character n-gram count vectorizer with `k=4` as the initial baseline. The vectorizer is fitted on training data only.

In [ ]:
KMER_SIZE = 4

vectorizer = CountVectorizer(
    analyzer="char",
    ngram_range=(KMER_SIZE, KMER_SIZE),
    lowercase=False,
    dtype=np.float32
)

X_train = vectorizer.fit_transform(train_df["sequence"])
X_test = vectorizer.transform(test_df["sequence"])
y_train = train_df["family"].to_numpy()
y_test = test_df["family"].to_numpy()

print("Training feature matrix:", X_train.shape)
print("Testing feature matrix:", X_test.shape)
print("Number of observed k-mer features:", len(vectorizer.get_feature_names_out()))
print("Training samples:", X_train.shape[0])
print("Number of k-mer features:", X_train.shape[1])
print("Test samples:", X_test.shape[0])
print("Number of classes:", len(np.unique(y_train)))
print("Feature matrix type:", type(X_train))


## 9. Train baseline classifiers
We compare five models: Multinomial Naive Bayes, Logistic Regression, Linear SVM, Decision Tree, and Random Forest. Class weighting is used for Logistic Regression, Linear SVM, Decision Tree, and Random Forest to reduce the tendency to ignore smaller classes. Naive Bayes does not provide the same `class_weight` option. Hyperparameter tuning will be a later step.


In [ ]:
models = {
    # 1. Probabilistic model
    "Multinomial Naive Bayes": MultinomialNB(
        alpha=0.5
    ),

    # 2. Linear models
    "Logistic Regression": LogisticRegression(
        max_iter=500,
        class_weight="balanced",
        solver="saga",
        tol=1e-3,
        random_state=RANDOM_STATE
    ),

    "Linear SVM": LinearSVC(
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    
    # 3. Decision Tree
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),

    # 4. Random Forest
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        max_features="sqrt",
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE
    )
}

fitted_models = {}
predictions = {}
results = []

for name, model in models.items():
    print(f"Training {name} ...")
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    fitted_models[name] = model
    predictions[name] = pred

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, pred, average="macro", zero_division=0
    )
    results.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1
    })

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
display(results_df)


In [ ]:
from sklearn.base import clone

parallel_models = {
    name: clone(model)
    for name, model in models.items()
}

parallel_models["Random Forest"].set_params(n_jobs=1)


In [ ]:
from joblib import Parallel, delayed
import time

# Train one model independently
def train_model_parallel(name, model, X_train, y_train, X_test):

    print(f"Starting: {name}")

    start = time.perf_counter()

    model.fit(X_train, y_train)

    training_time = time.perf_counter() - start

    start = time.perf_counter()

    predictions = model.predict(X_test)

    prediction_time = time.perf_counter() - start

    print(f"Completed: {name}")

    return name, model, predictions, training_time, prediction_time


# Execute models in parallel
parallel_start = time.perf_counter()

parallel_results = Parallel(
    n_jobs=5,
    backend="loky",
    verbose=10
)(
    delayed(train_model_parallel)(
        name, model, X_train, y_train, X_test
    )
    for name, model in parallel_models.items()
)

parallel_total_time = time.perf_counter() - parallel_start

print("\nTotal parallel execution time:",
      round(parallel_total_time, 2), "seconds")


In [ ]:
parallel_fitted_models = {}
parallel_predictions = {}
parallel_metrics = []

for name, model, pred, train_time, predict_time in parallel_results:

    parallel_fitted_models[name] = model
    parallel_predictions[name] = pred

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test,
        pred,
        average="macro",
        zero_division=0
    )

    parallel_metrics.append({
        "model": name,
        "accuracy": accuracy_score(y_test, pred),
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "training_time_sec": round(train_time, 2),
        "prediction_time_sec": round(predict_time, 2)
    })

parallel_results_df = pd.DataFrame(parallel_metrics)

display(
    parallel_results_df.sort_values(
        "macro_f1",
        ascending=False
    )
)


In [ ]:
# The original sequential cell did not record per-model timings.
# Therefore, sum the individual durations measured in the parallel run to
# estimate how long those same tasks would take if run one after another.
estimated_sequential_time = (
    parallel_results_df["training_time_sec"].sum()
    + parallel_results_df["prediction_time_sec"].sum()
)

print("Estimated sequential model execution time:",
      round(estimated_sequential_time, 2), "seconds")
print("Actual parallel wall-clock execution time:",
      round(parallel_total_time, 2), "seconds")

if parallel_total_time > 0:
    estimated_speedup = estimated_sequential_time / parallel_total_time
    print("Estimated speedup:", round(estimated_speedup, 2), "x")

print("\nNote: This is an estimate, not a direct comparison with the earlier sequential run.")
print("Parallel timings include process startup/coordination overhead; model runtimes can also vary between runs.")


## 10. Inspect the selected baseline model
For this initial experiment, the model with the highest macro-F1 in `results_df` is selected and its classification report is displayed. **Important:** the same test set was used to compare the five models, so this report is exploratory rather than an unbiased final estimate of generalisation performance. For a final evaluation, introduce a separate validation set (or cross-validation) for model selection and reserve the test set for one final evaluation.


In [ ]:
BEST_MODEL_NAME = results_df.iloc[0]["model"]
best_model = fitted_models[BEST_MODEL_NAME]
best_predictions = predictions[BEST_MODEL_NAME]

print("Selected baseline (highest test macro-F1; exploratory selection):", BEST_MODEL_NAME)
print(classification_report(
    y_test,
    best_predictions,
    labels=sorted(np.unique(y_test)),
    zero_division=0
))


## 11. Build a record-level prediction and error table
The model is trained on unique sequences. We attach the original BOLD record identifiers and metadata to predictions so individual errors can be investigated later. The table distinguishes the actual family, predicted family, and correctness. Per-family TP/FP/FN/TN are computed in a one-vs-rest manner.

In [ ]:
prediction_table = test_df[[
    "processid", "sampleid", "family", "genus", "species",
    "country/ocean", "province/state", "collection_date_start",
    "habitat", "sequence", "sequence_length", "ambiguous_base_count"
]].copy()

prediction_table = prediction_table.rename(columns={"family": "actual_family"})
prediction_table["predicted_family"] = best_predictions
prediction_table["correct"] = (
    prediction_table["actual_family"] == prediction_table["predicted_family"]
)
prediction_table["error_type_overall"] = np.where(
    prediction_table["correct"], "correct", "misclassified"
)

fp_fn_records = prediction_table.loc[~prediction_table["correct"]].copy()
print("Test records:", len(prediction_table))
print("Correct:", int(prediction_table["correct"].sum()))
print("Misclassified:", len(fp_fn_records))
display(fp_fn_records.head(10))


### One-vs-rest TP, TN, FP and FN for every family

In [ ]:
all_families = sorted(model_df["family"].unique())
y_true_arr = prediction_table["actual_family"].to_numpy()
y_pred_arr = prediction_table["predicted_family"].to_numpy()

confusion_rows = []
for family in all_families:
    actual_positive = (y_true_arr == family)
    predicted_positive = (y_pred_arr == family)
    tp = int(np.sum(actual_positive & predicted_positive))
    tn = int(np.sum(~actual_positive & ~predicted_positive))
    fp = int(np.sum(~actual_positive & predicted_positive))
    fn = int(np.sum(actual_positive & ~predicted_positive))
    confusion_rows.append({
        "family": family, "TP": tp, "TN": tn, "FP": fp, "FN": fn
    })

per_family_confusion = pd.DataFrame(confusion_rows)
display(per_family_confusion.sort_values(["FN", "FP"], ascending=False).head(25))


## 12. Export the baseline outputs
These CSVs will be useful for the next stage: detailed FP/FN investigation with SHAP-style feature explanations, sequence alignment/similarity, class-frequency context, and record-quality checks.

In [ ]:
OUTPUT_DIR = Path("family_classification_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

family_counts_all.to_csv(OUTPUT_DIR / "family_record_counts.csv", index=False)
results_df.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
prediction_table.drop(columns=["sequence"]).to_csv(
    OUTPUT_DIR / "test_predictions.csv", index=False
)
fp_fn_records.to_csv(OUTPUT_DIR / "misclassified_records_for_investigation.csv", index=False)
per_family_confusion.to_csv(OUTPUT_DIR / "per_family_TP_TN_FP_FN.csv", index=False)
coi[["processid", "family", "marker_code", "sequence_length",
     "ambiguous_base_count", "audit_status"]].to_csv(
    OUTPUT_DIR / "record_quality_audit.csv", index=False
)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Files:", [p.name for p in OUTPUT_DIR.iterdir()])


## 13. Interpretation notes and next steps

### What this first notebook does
- Keeps all family labels; it does not drop rare families using a minimum-count threshold.
- Excludes only records that cannot be used in the initial A/C/G/T k-mer baseline, and flags them for audit.
- Removes exact duplicate sequences from the modelling table so repeated copies do not dominate the baseline.
- Flags identical sequences associated with conflicting family labels rather than arbitrarily choosing one label.
- Keeps single-sequence families in training and reports that they cannot be independently evaluated in a held-out test set.
- Produces record-level predictions and one-vs-rest TP/TN/FP/FN counts.

### Important limitations
1. **Rare families:** A family with one unique sequence cannot be evaluated on unseen examples. Keeping the class does not create information that the dataset does not contain.
2. **Species leakage:** The current split groups exact identical sequences, but different sequences from the same species may still occur in both sets. A species-grouped evaluation is a useful next experiment.
3. **Biological interpretation:** SHAP explains model behaviour, not biological causation. Sequence alignment and label verification are needed to investigate biological explanations.
4. **Confidence:** Linear SVM does not directly provide calibrated probabilities. We will add calibrated confidence estimates in a later notebook stage if needed.
5. **Metadata:** Metadata is retained for error analysis, not used as model input in the primary DNA-only experiment.
6. **Model selection:** The five models were compared using the test set in this baseline. Use a separate validation set or cross-validation for model selection, then evaluate the chosen model once on an untouched test set.

### Next stage
After reviewing the family counts, split coverage, model scores, and misclassified records, extend the notebook with:
- a train/validation/test design for model selection and final evaluation;
- species-grouped validation;
- SHAP or model-specific feature explanations;
- nearest-neighbour sequence similarity / pairwise alignment;
- systematic FP/FN reason analysis;
- an optional, carefully controlled DNA + metadata experiment.